# 8.2 Institutional Pattern Discovery II: Unsupervised Learning with Survey Data

## Introduction



In the previous sections, we used **supervised** models to predict known outcomes like retention or stress. However, some of the most valuable institutional insights come from **unsupervised learning**, where we don't have a specific "label" to predict.

Instead, we ask the data to reveal the natural, hidden groupings within our student population. This allows us to move beyond broad demographic categories and discover **Student Personas** based on their actual behaviors and voices.

<br>

#### **Guiding Questions**
1. Can we discover natural groupings of students by combining structured academic records with unstructured survey text?
2. How do we interpret and translate these mathematical "clusters" into actionable institutional strategies?

<br>

#### **Learning Objectives**
By the end of this notebook, you will be able to:
* **Preprocess** a mixed dataset (numeric, categorical, and text) into a unified, model-ready matrix.
* **Apply K-Means Clustering** to find latent student segments.
* **Determine the Optimal K** using the Elbow Method (Inertia).
* **Profile and Interpret** clusters to communicate high-impact findings to campus stakeholders.


## 1. Why Unsupervised Learning in IR?


Supervised learning tells us *who* is likely to leave or succeed. Unsupervised learning tells us **who our students actually are**.

By using **K-Means Clustering**, we can surface distinct segments that might otherwise remain hidden in large datasets, such as:
* **The High-Engagement Thrivers:** Students with high GPAs and positive survey sentiment who may be perfect candidates for peer-mentorship roles.
* **The "Silent" At-Risk Group:** Students with moderate grades but low survey engagement: a subtle early-warning signal that administrative data often misses.
* **The Transition-Strugglers:** First-gen or transfer students whose open-ended comments reflect high academic anxiety despite stable mid-term grades.

> **Institutional Perspective:** Clustering turns "Big Data" into "Human Personas." Instead of looking at 5,000 individual rows, we can look at 3 or 4 meaningful student profiles, allowing for much more personalized and effective resource allocation.

## 2. Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import random
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.options.display.max_columns = None
np.random.seed(42)
random.seed(42)

filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 2 v2/data/'
df_training = pd.read_csv(f'{filepath}training.csv')

df_training19 = df_training[df_training['COHORT'] == 'Fall 2019'].copy()
print("Cohort size:", len(df_training19))
df_training19


Cohort size: 5162


,SID,COHORT,RACE_ETHNICITY,GENDER,FIRST_GEN_STATUS,HS_GPA,HS_MATH_GPA,HS_ENGL_GPA,COLLEGE,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,UNITS_ATTEMPTED_3,UNITS_COMPLETED_1,UNITS_COMPLETED_2,UNITS_COMPLETED_3,DFW_UNITS_1,DFW_UNITS_2,DFW_UNITS_3,GPA_1,GPA_2,GPA_3,SEM_3_STATUS,DFW_RATE_1,DFW_RATE_2,GRADE_POINTS_1,GRADE_POINTS_2,GRADE_POINTS_3
5,UHKR1SFYP,Fall 2019,Asian,Male,Continuing Generation,3.500,2.667,4.167,Letters & Humanities,16.0,15.0,9.0,16.0,15.0,12.0,0.0,0.0,0.0,3.437500,3.800000,3.000000,E,0.00,0.000000,55.0,57.0,27.0
8,UHNF02H9O,Fall 2019,Hispanic,Female,First Generation,2.944,4.000,3.500,General Studies,15.0,13.0,12.0,15.0,13.0,15.0,0.0,0.0,0.0,2.600000,4.000000,3.750000,E,0.00,0.000000,39.0,52.0,45.0
9,UHOFN46WE,Fall 2019,Black or African American,Female,Continuing Generation,4.069,3.400,4.125,Business Administration,12.0,12.0,3.0,12.0,15.0,15.0,0.0,0.0,0.0,3.500000,4.000000,4.000000,E,0.00,0.000000,42.0,48.0,12.0
10,UHP7B9BTV,Fall 2019,Hispanic,Female,First Generation,3.032,1.250,2.625,Business Administration,12.0,9.0,12.0,12.0,6.0,0.0,3.0,6.0,12.0,1.500000,2.000000,0.750000,E,0.00,0.333333,18.0,18.0,9.0
11,UI30HVLIL,Fall 2019,Hispanic,Female,First Generation,3.273,3.250,2.800,Business Administration,15.0,13.0,16.0,15.0,13.0,13.0,0.0,3.0,3.0,3.200000,3.076923,2.625000,E,0.00,0.000000,48.0,40.0,42.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19832,N26B5Y6HE,Fall 2019,Nonresident alien,Female,First Generation,3.281,3.000,3.667,Letters & Humanities,16.0,9.0,12.0,16.0,9.0,15.0,0.0,0.0,0.0,2.937500,4.000000,3.250000,E,0.00,0.000000,47.0,36.0,39.0
19834,N28LHIN72,Fall 2019,White,Female,Continuing Generation,3.621,2.800,3.667,Natural and Mathematical Sciences,13.0,13.0,15.0,13.0,14.0,15.0,0.0,0.0,0.0,2.769231,3.769231,3.066667,E,0.00,0.000000,36.0,49.0,46.0
19835,N299IIGIN,Fall 2019,Two or More Races,Female,Continuing Generation,4.094,3.667,4.167,Visual & Performing Arts,15.0,15.0,12.0,15.0,16.0,13.0,0.0,0.0,0.0,4.000000,4.000000,4.000000,E,0.00,0.000000,60.0,60.0,48.0
19837,N2C5YLPI1,Fall 2019,Asian,Male,Continuing Generation,3.658,3.750,3.167,Engineering & Technology,16.0,16.0,16.0,12.0,3.0,16.0,4.0,13.0,0.0,2.125000,1.125000,3.500000,E,0.25,0.812500,34.0,18.0,56.0



## 3. Preprocessing for Cluster Analysis


As established earlier in the certificate program, we use the `ColumnTransformer` to handle mixed data types. However, for **K-Means Clustering**, this step is even more critical because the algorithm is entirely "distance-based."

If we do not scale, features with larger raw numbers (like Units Attempted) will mathematically "overwhelm" more nuanced metrics (like GPA), leading to clusters that only reflect credit loads rather than holistic student profiles.

| Feature Category | Scaling Strategy | IR Strategy for Clustering |
| :--- | :--- | :--- |
| **GPA & DFW Rates** | **MinMaxScaler** | Keeps academic performance within a comparable 0.0–1.0 range. |
| **Credit Volume** | **StandardScaler** | Normalizes "Units Attempted" so they don't dominate the distance calculation. |
| **Demographics** | **OneHotEncoder** | Translates categorical identity into numerical coordinates for the model. |

<br>

> **Instructor Perspective:** In Course 2, you learned the *code* for this pipeline. In Course 3, focus on the **Impact**. Scaling ensures that a 0.5-point difference in GPA is treated with the same mathematical "weight" as a 3-unit difference in course load.


In [ ]:
minmax_cols = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2']
standard_cols = ['UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
categorical_cols = ['GENDER', 'RACE_ETHNICITY', 'FIRST_GEN_STATUS']

# Drop rows missing any of these key columns
df19 = df_training19.dropna(subset=minmax_cols + standard_cols + categorical_cols).copy()
print(f"Rows after dropping incomplete records: {len(df19)}")

preprocessor = ColumnTransformer(
    transformers=[
        ('minmax',   MinMaxScaler(), minmax_cols),
        ('standard', StandardScaler(), standard_cols),
        ('onehot',   OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ],
    remainder='drop'
)

X_structured = preprocessor.fit_transform(df19)
df_structured = pd.DataFrame(X_structured, index=df19.index)
print("Structured feature matrix:", df_structured.shape)


Rows after dropping incomplete records: 5162
Structured feature matrix: (5162, 19)



## 4. Reduce TF-IDF Text Features with PCA


Free-response text vectorized with TF-IDF produces hundreds of columns, far too many to cluster directly alongside our ~19 structured features. To prevent the text data from mathematically "overwhelming" our academic metrics, we use **PCA (Principal Component Analysis)**.

PCA compresses the high-dimensional word matrix into a small number of "Principal Components" that capture the core variance (the "essence") of the student voice.

**Determining the Optimal Number of Components:**


We use an **elbow plot** (Cumulative Explained Variance) to choose our cutoff. In institutional research, capturing **80% of the variance** is a common standard, as it retains the primary emotional and topical signals while discarding the "noise" of rare or unique words.

<br>

> **Instructor Perspective:** Think of PCA as an "Equalizer." Without it, the 250+ word columns would act like 250+ separate votes in the clustering algorithm, while your GPA only gets one vote. PCA condenses those 250+ votes into roughly 30, giving the "Student Voice" and "Student Records" a more balanced influence on the final personas.



In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
ML_Survey_Data19_Num = pd.read_csv(f'{filepath}ML_Survey_Data19_Num.csv')
display(ML_Survey_Data19_Num)

,SID,NSSE_Q1,NSSE_Q2,NSSE_Q3,NSSE_Q4,NSSE_Q5,NSSE_Q6,NSSE_Q7,NSSE_Q8,NSSE_Q9,NSSE_Q10,ability,academic,academically,access,adapting,advance,advanced,affects,ahead,analysis,apply,appreciate,approaches,appropriately,areas,asking,aspects,assignment,assignments,attending,balancing,basics,best,better,build,busy,campus,carefully,catching,challenge,challenges,challenging,chosen,clarifying,class,classes,classmates,classroom,clear,clearer,click,coaching,college,comfortable,comments,compare,concepts,concrete,confidence,confident,connect,consistently,constraints,content,course,courses,coursework,day,deadlines,deeper,depends,determined,develop,discovering,discussions,doable,don,eager,earlier,early,easy,education,embarrassed,engaged,enjoy,especially,exams,excellence,excitement,exist,exists,expect,expectations,expected,experience,explore,fall,falling,family,far,fast,faster,feedback,feel,feeling,feels,felt,field,figure,figuring,finding,fine,finish,focus,footing,forward,foundational,fully,going,group,groups,grow,guidance,habits,hard,harder,heavy,help,helpful,helps,hesitate,higher,hit,hope,hours,improve,instead,instructor,instructors,interesting,journey,juggling,just,keeping,keeps,know,larger,late,learning,level,life,like,looking,lost,lot,make,makes,manage,managing,material,meet,mix,moments,motivated,moves,multiple,need,new,notes,office,organized,overall,overwhelmed,pace,participating,passionate,path,performance,personal,pile,plan,planning,positive,practice,prepared,prioritize,priority,problems,projects,pushes,questions,quickly,reach,refine,regularly,remember,require,resources,responsibilities,rest,results,review,reviewing,rewarding,rhythm,rigor,rigorous,routine,rubrics,rushing,scheduling,school,seeing,seeking,skills,soon,speak,specific,standard,start,stay,steps,strategies,stressed,strive,strong,structure,struggle,studies,study,studying,succeed,support,sure,survive,takes,taking,talking,team,tend,term,things,think,thinking,time,times,topics,tougher,track,transition,trying,tuning,turning,tutoring,understand,use,used,useful,usually,ve,want,way,week,weeks,wish,work,working,workload,writing
0,UHKR1SFYP,2,3,2,2,2,2,2,2,3,2,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.457811,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.402039,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000,0.0000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00000,0.000000,0.0,0.000000,0.00000,0.0,0.000000,0.0,0.0,0.457811,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.00000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.457811,0.0,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.457811,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0
1,UHNF02H9O,2,2,1,1,1,2,1,1,1,1,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.309518,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.00

In [ ]:
tfidf_matrix = ML_Survey_Data19_Num.iloc[:,11:]
print("TF-IDF matrix:", tfidf_matrix.shape)


TF-IDF matrix: (5162, 259)


We use an elbow plot to identify the minimum number of components needed to capture 80% of the text variance, effectively filtering out linguistic 'noise' while preserving the core student voice.

In [ ]:
# Choose number of PCA components by explained variance
pca_full = PCA(random_state=42)
pca_full.fit(tfidf_matrix)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
threshold = 0.80
n_components = int(np.searchsorted(cumvar, threshold)) + 1

fig = px.line(
    x=range(1, len(cumvar) + 1), y=cumvar,
    labels={'x': 'Number of PCA Components', 'y': 'Cumulative Explained Variance'},
    title='PCA Explained Variance — TF-IDF Features'
)
fig.add_hline(y=threshold, line_dash='dash', annotation_text=f'{int(threshold*100)}% threshold')
fig.add_vline(x=n_components, line_dash='dot',
              annotation_text=f'{n_components} components', annotation_position='top right')
fig.show()
print(f"→ Using {n_components} PCA components to capture {threshold*100:.0f}% of text variance")

→ Using 33 PCA components to capture 80% of text variance


Once the optimal threshold is found, we project the 250+ raw word columns into 33 Principal Components (`TEXT_PC1`, `TEXT_PC2`, etc.) to create a balanced feature set for clustering.

In [ ]:
# Apply PCA with the chosen number of components
pca = PCA(n_components=n_components, random_state=42)
X_text_pca = pca.fit_transform(tfidf_matrix)
df_text_pca = pd.DataFrame(X_text_pca, index=df19.index,
                           columns=[f'TEXT_PC{i+1}' for i in range(n_components)])
print("Text PCA matrix:", df_text_pca.shape)


Text PCA matrix: (5162, 33)


## 5. Combine All Features




We use `pd.concat` with `axis=1` to join our structured features and our compressed text components side-by-side.

> **Technical Note:** We cast all column names to strings at this stage. This is a requirement for the K-Means algorithm, which expects a consistent header format to perform distance calculations.

In [ ]:
df_all = pd.concat([df_structured, df_text_pca], axis=1)
df_all.columns = df_all.columns.astype(str)  # KMeans requires string column names
print("Combined feature matrix:", df_all.shape)
df_all.head(3)


Combined feature matrix: (5162, 52)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,TEXT_PC1,TEXT_PC2,TEXT_PC3,TEXT_PC4,TEXT_PC5,TEXT_PC6,TEXT_PC7,TEXT_PC8,TEXT_PC9,TEXT_PC10,TEXT_PC11,TEXT_PC12,TEXT_PC13,TEXT_PC14,TEXT_PC15,TEXT_PC16,TEXT_PC17,TEXT_PC18,TEXT_PC19,TEXT_PC20,TEXT_PC21,TEXT_PC22,TEXT_PC23,TEXT_PC24,TEXT_PC25,TEXT_PC26,TEXT_PC27,TEXT_PC28,TEXT_PC29,TEXT_PC30,TEXT_PC31,TEXT_PC32,TEXT_PC33
5,0.622606,0.859375,0.95,0.0,0.0,1.204860,0.638758,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.023613,-0.103361,0.057734,0.015776,-0.221725,-0.487695,0.050122,0.773817,0.151532,-0.088755,0.031241,0.097916,-0.041674,-0.050040,0.025866,-0.018332,-0.020864,-0.091023,-0.035653,-0.002261,0.008150,0.082014,0.013345,-0.017664,-0.003739,-0.010717,-0.024523,-0.021465,-0.024537,-0.004703,-0.028791,0.026912,0.011032
8,0.362915,0.650000,1.00,0.0,0.0,0.806635,-0.069723,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.067406,0.051860,0.072209,-0.016120,-0.035218,-0.119566,0.116640,0.167224,0.041116,-0.053032,-0.062078,-0.068696,-0.003302,0.146157,-0.117805,0.061409,0.014751,0.061041,0.347019,0.001106,-0.062652,-0.295891,-0.244583,0.237964,0.226689,0.117573,0.073672,0.156698,0.208002,-0.012112,0.219897,-0.215340,-0.213052
9,0.888370,0.875000,1.00,0.0,0.0,-0.388043,-0.423963,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.016689,-0.048284,-0.018743,-0.023599,-0.051865,-0.028445,-0.063667,0.004296,-0.046113,-0.042212,-0.036300,-0.054128,0.030232,0.037774,0.043003,0.044613,-0.052349,0.185471,-0.031442,0.125228,0.152910,-0.161609,-0.133176,-0.089858,-0.098904,0.138702,0.021644,0.221073,-0.118036,-0.183125,0.154070,0.373265,0.303613


When we used the `ColumnTransformer` earlier, it added prefixes like `minmax__` and `onehot__` to our variables. While useful for tracking our steps, these make for cluttered visualizations and reports.

In this step, we programmatically strip those prefixes to restore clean, intuitive headers (e.g., changing `minmax__HS_GPA` back to `HS_GPA`). This ensures our final cluster profiles are easy for institutional stakeholders to read and interpret.

In [ ]:
transformed_structured_feature_names = preprocessor.get_feature_names_out()

cleaned_structured_cols = []
for col_name in transformed_structured_feature_names:
    if col_name.startswith('minmax__'):
        cleaned_structured_cols.append(col_name.replace('minmax__', ''))
    elif col_name.startswith('standard__'):
        cleaned_structured_cols.append(col_name.replace('standard__', ''))
    elif col_name.startswith('onehot__'):
        # Remove the 'onehot__' prefix to make names like 'GENDER_Female' cleaner
        cleaned_structured_cols.append(col_name.replace('onehot__', ''))
    else:
        cleaned_structured_cols.append(col_name)

# The PCA column names are already correctly named in df_text_pca.columns
pca_cols = df_text_pca.columns.tolist()

# Combine all column names
df_all.columns = cleaned_structured_cols + pca_cols

print("Combined feature matrix with renamed columns:", df_all.shape)
display(df_all.head(3))

ML_Survey_Data19 = df_all

ML_Survey_Data19

Combined feature matrix with renamed columns: (5162, 52)


,HS_GPA,GPA_1,GPA_2,DFW_RATE_1,DFW_RATE_2,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,GENDER_Female,GENDER_Male,RACE_ETHNICITY_Asian,RACE_ETHNICITY_Black or African American,RACE_ETHNICITY_Hispanic,RACE_ETHNICITY_Nonresident alien,RACE_ETHNICITY_Other,RACE_ETHNICITY_Two or More Races,RACE_ETHNICITY_White,FIRST_GEN_STATUS_Continuing Generation,FIRST_GEN_STATUS_First Generation,FIRST_GEN_STATUS_Unknown,TEXT_PC1,TEXT_PC2,TEXT_PC3,TEXT_PC4,TEXT_PC5,TEXT_PC6,TEXT_PC7,TEXT_PC8,TEXT_PC9,TEXT_PC10,TEXT_PC11,TEXT_PC12,TEXT_PC13,TEXT_PC14,TEXT_PC15,TEXT_PC16,TEXT_PC17,TEXT_PC18,TEXT_PC19,TEXT_PC20,TEXT_PC21,TEXT_PC22,TEXT_PC23,TEXT_PC24,TEXT_PC25,TEXT_PC26,TEXT_PC27,TEXT_PC28,TEXT_PC29,TEXT_PC30,TEXT_PC31,TEXT_PC32,TEXT_PC33
5,0.622606,0.859375,0.95,0.0,0.0,1.204860,0.638758,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.023613,-0.103361,0.057734,0.015776,-0.221725,-0.487695,0.050122,0.773817,0.151532,-0.088755,0.031241,0.097916,-0.041674,-0.050040,0.025866,-0.018332,-0.020864,-0.091023,-0.035653,-0.002261,0.008150,0.082014,0.013345,-0.017664,-0.003739,-0.010717,-0.024523,-0.021465,-0.024537,-0.004703,-0.028791,0.026912,0.011032
8,0.362915,0.650000,1.00,0.0,0.0,0.806635,-0.069723,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.067406,0.051860,0.072209,-0.016120,-0.035218,-0.119566,0.116640,0.167224,0.041116,-0.053032,-0.062078,-0.068696,-0.003302,0.146157,-0.117805,0.061409,0.014751,0.061041,0.347019,0.001106,-0.062652,-0.295891,-0.244583,0.237964,0.226689,0.117573,0.073672,0.156698,0.208002,-0.012112,0.219897,-0.215340,-0.213052
9,0.888370,0.875000,1.00,0.0,0.0,-0.388043,-0.423963,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.016689,-0.048284,-0.018743,-0.023599,-0.051865,-0.028445,-0.063667,0.004296,-0.046113,-0.042212,-0.036300,-0.054128,0.030232,0.037774,0.043003,0.044613,-0.052349,0.185471,-0.031442,0.125228,0.152910,-0.161609,-0.133176,-0.089858,-0.098904,0.138702,0.021644,0.221073,-0.118036,-0.183125,0.154070,0.373265,0.303613



## 6. Choose Number of Clusters: Elbow Method


K-Means requires us to specify $K$ (the number of clusters) upfront. Because there is no "correct" label in unsupervised learning, we use the **Elbow Method** to find a mathematically sound value for $K$.

We plot **Inertia** (the sum of squared distances from each student point to their assigned cluster center). As $K$ increases, inertia naturally drops. We are looking for the "Elbow"—the specific point where adding more clusters no longer yields a significant decrease in inertia.



> **Instructor Perspective:** In Institutional Research, choosing $K$ is a balance between **Granularity** and **Actionability**. A $K$ of 20 might be mathematically precise, but a Provost cannot design 20 different student success initiatives. We typically look for $K$ values between 3 and 6 to ensure the resulting personas are distinct enough to drive real campus policy.


We iterate through a range of potential cluster counts ($K=1$ to $10$), calculating the inertia at each step to visualize the trade-off between model complexity and group cohesion.

In [ ]:
inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df_all)
    inertia.append(km.inertia_)

fig = px.line(x=list(K_range), y=inertia,
              markers=True,
              labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'},
              title='Elbow Method — Choosing K for K-Means')
fig.show()
print("Look for the point where the curve flattens — that is your elbow.")


Look for the point where the curve flattens — that is your elbow.


## 7. Fit K-Means & Assign Cluster Labels





Now that we have identified the "Elbow" at **K=3**, we fit our final model. This process assigns every student in our Fall 2019 cohort to one of three distinct groups based on the mathematical similarities in their academic records and survey voices.

> **Technical Note:** We use `n_init=10` to ensure the model runs multiple times with different starting points, selecting the best result to avoid "local minima" (sub-optimal groupings).


In [ ]:
OPTIMAL_K = 3  # ← update this based on the elbow plot above

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
kmeans.fit(df_all)

df19 = df19.copy()
df19['Cluster'] = kmeans.labels_

print("Cluster sizes:")
print(df19['Cluster'].value_counts().sort_index())


Cluster sizes:
Cluster
0    2687
1    1781
2     694
Name: count, dtype: int64


## 8. Visualize Clusters with PCA



Our master matrix has **52 dimensions** (19 structured features + 33 text components). Because the human eye cannot see in 52D, we use PCA once more to project the data down to **two coordinates (PC1 and PC2)** for visualization.

**Important Distinction:** * The **Clustering** was done using all 52 features to ensure maximum accuracy.
* The **Visualization** uses only 2 features so we can "see" the separation between our student groups.



> **Key note:** Look for the "overlap" in the scatter plot. In social science data, clusters are rarely perfectly isolated circles; students are complex and often sit on the borders between personas. Our goal is to find the **dominant trend** in each group.



In [ ]:
pca2 = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(df_all)

df_plot = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'Cluster': df19['Cluster'].astype(str),
    'GPA': df19['HS_GPA'].round(2),
    'First_Gen': df19['FIRST_GEN_STATUS'],
    'Gender': df19['GENDER']
}, index=df19.index)

fig = px.scatter(
    df_plot, x='PC1', y='PC2',
    color='Cluster',
    hover_data=['GPA', 'First_Gen', 'Gender'],
    title=f'K-Means Clusters (K={OPTIMAL_K}) — Visualized with 2D PCA',
    labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'}
)
fig.update_traces(marker=dict(size=6, opacity=0.75))
fig.show()



## 9. Profile and Interpret Clusters


The scatter plot shows *where* clusters sit in 2D space, but to take action, we must understand *who* is in each group. We "profile" these clusters by calculating the average values for our key academic and demographic features.

In Institutional Research, this step is known as **Persona Development**. We are looking for the "Typical Student" within each cluster so we can tailor support services to their specific needs.



> **Instructor Perspective:** Don't just look at the numbers—look for the **Story**. If Cluster 2 has a high DFW rate and mentions "work-life balance" in their comments, they aren't just a "low-performing group"; they are a "High-Obligation" group that may need more flexible course scheduling.


We calculate the mean for each academic metric by cluster to identify the 'Performance Fingerprint' of each student group.

In [ ]:
# Numeric profile
profile_cols = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2',
                'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
profile_cols = [c for c in profile_cols if c in df19.columns]

cluster_profile = df19.groupby('Cluster')[profile_cols].mean().round(3)
print("── Cluster Numeric Profile ──")
cluster_profile


── Cluster Numeric Profile ──


,HS_GPA,GPA_1,GPA_2,DFW_RATE_1,DFW_RATE_2,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2
Cluster,,,,,,,
0,3.670,3.134,3.223,0.078,0.080,14.636,14.274
1,3.576,2.839,3.025,0.122,0.121,10.545,13.670
2,3.476,2.589,2.496,0.190,0.254,12.774,7.810


By normalizing the distributions of Gender and First-Gen status, we can see if certain demographic groups are disproportionately represented in specific success clusters.

In [ ]:
# Categorical breakdown
print("── First-Generation Status by Cluster ──")
print(pd.crosstab(df19['Cluster'], df19['FIRST_GEN_STATUS'], normalize='index').round(3))

print("\n── Gender by Cluster ──")
print(pd.crosstab(df19['Cluster'], df19['GENDER'], normalize='index').round(3))


── First-Generation Status by Cluster ──
FIRST_GEN_STATUS  Continuing Generation  First Generation  Unknown
Cluster                                                           
0                                 0.669             0.271    0.060
1                                 0.563             0.321    0.116
2                                 0.607             0.298    0.095

── Gender by Cluster ──
GENDER   Female   Male
Cluster               
0         0.585  0.415
1         0.606  0.394
2         0.540  0.460


Finally, we pull raw survey text from each cluster to hear the 'Student Voice' behind the numbers, allowing us to validate the mathematical groupings with qualitative context.

In [ ]:
# Sample comments from each cluster
print("── Representative Comments by Cluster ──")
for c in sorted(df19['Cluster'].unique()):
    subset = df19[df19['Cluster'] == c]
    examples = subset.sample(min(2, len(subset)), random_state=42)['Free_Response_Text'].tolist()
    print(f"\nCluster {c} (n={len(subset)}):")
    for ex in examples:
        print(f"  • {ex}")


── Representative Comments by Cluster ──


KeyError: 'Free_Response_Text'

In [ ]:
# Visualize numeric profile as a heatmap
import plotly.figure_factory as ff

z = cluster_profile.values.tolist()
x = cluster_profile.columns.tolist()
y = [f'Cluster {i}' for i in cluster_profile.index]

fig = px.imshow(
    cluster_profile,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='Blues',
    title='Cluster Profiles — Average Feature Values',
    labels={'x': 'Feature', 'y': 'Cluster', 'color': 'Mean Value'}
)
fig.show()


## 10. Wrap-Up



**What you did:**
- Preprocessed a mixed dataset (numeric + categorical + free-text) into a single feature matrix
- Used **PCA** to compress TF-IDF text vectors into a manageable number of components
- Applied the **Elbow Method** to choose a reasonable K
- Fit **K-Means** and assigned each student a cluster label
- Profiled clusters by GPA, DFW rates, demographic composition, and survey tone

**How to use clusters in IR practice:**
- Flag clusters with low GPA + high DFW for early-alert advising outreach
- Identify clusters dominated by first-gen students who express struggle — target programming
- Track cluster membership over cohorts to evaluate intervention effectiveness

**Key caution:** Clusters are patterns, not ground truth. Always validate with domain expertise and be cautious about using cluster labels in high-stakes decisions without further analysis.

